<a href="https://colab.research.google.com/github/jojojohnson013/AIML_Internship/blob/main/sentimental_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [2]:
vocab_size = 10000
#Keep only the top 10000 most frequent words
(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Testing samples: 25000


In [3]:
max_length = 200

X_train = pad_sequences(
    X_train,
    maxlen=max_length,
    padding='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=max_length,
    padding='post'
)


In [4]:
model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length
    ),

    SimpleRNN(64),

    Dense(32, activation='relu'),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 95ms/step - accuracy: 0.5047 - loss: 0.6941 - val_accuracy: 0.5132 - val_loss: 0.6920
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 26s 84ms/step - accuracy: 0.5324 - loss: 0.6883 - val_accuracy: 0.5078 - val_loss: 0.6955
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 85ms/step - accuracy: 0.5515 - loss: 0.6780 - val_accuracy: 0.5466 - val_loss: 0.6707
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 44s 94ms/step - accuracy: 0.6003 - loss: 0.6346 - val_accuracy: 0.5578 - val_loss: 0.7031
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 36s 79ms/step - accuracy: 0.6330 - loss: 0.5778 - val_accuracy: 0.5812 - val_loss: 0.6650


In [8]:
loss, accuracy = model.evaluate(
    X_test,
    y_test
)

print("Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.5852 - loss: 0.6646
Accuracy: 0.5851600170135498


In [9]:
from tensorflow.keras.datasets import imdb

word_index = imdb.get_word_index()

# Shift indices by 3 because Keras reserves 0,1,2
word_index = {k: (v + 3) for k, v in word_index.items()}

word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2
word_index["<UNUSED>"] = 3

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [10]:
def review_to_sequence(review):

    review = review.lower().split()

    sequence = []

    for word in review:
        sequence.append(word_index.get(word, 2))  # 2 = unknown word

    return sequence

In [11]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

max_length = 200

def predict_sentiment(review):

    sequence = review_to_sequence(review)

    padded = pad_sequences(
        [sequence],
        maxlen=max_length,
        padding='post'
    )

    prediction = model.predict(padded, verbose=0)

    score = prediction[0][0]

    print("Sentiment Score:", score)

    if score > 0.5:
        print("Positive Review ")
    else:
        print("Negative Review ")

In [12]:
predict_sentiment(
    "This movie was amazing and the acting was excellent"
)

Sentiment Score: 0.5588603
Positive Review 


In [13]:
predict_sentiment(
    "i dont like this movie "
)

Sentiment Score: 0.55886024
Positive Review 


IMDB DATASET

In [14]:
import pandas as pd

df = pd.read_csv("/content/IMDB Dataset.csv", encoding='latin1')

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [15]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [16]:
import re

negation_words = [
    "not good", "not bad", "not great", "don't like",
    "didn't like", "never liked", "wasn't good",
    "isn't good", "no good"
]

def clean_text(text):
    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s']", " ", text)

    # convert negations into single tokens
    for phrase in negation_words:
        text = text.replace(phrase, phrase.replace(" ", "_"))

    return text

| Original phrase | Converted    |
| --------------- | ------------ |
| "not good"      | "not_good"   |
| "don't like"    | "don't_like" |


eg:This movie is not good at all

after transformation :This movie is not_good at all

In [17]:
df['review'] = df['review'].apply(clean_text)

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

Every review is converted into a list of numbers.

In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size = 20000
max_len = 250

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),

    LSTM(128, dropout=0.3, recurrent_dropout=0.3),

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 415s 819ms/step - accuracy: 0.5414 - loss: 0.6773 - val_accuracy: 0.5940 - val_loss: 0.6216
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 446s 827ms/step - accuracy: 0.5980 - loss: 0.6114 - val_accuracy: 0.5971 - val_loss: 0.6242
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 436s 872ms/step - accuracy: 0.6813 - loss: 0.5637 - val_accuracy: 0.8515 - val_loss: 0.3788
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 445s 890ms/step - accuracy: 0.8887 - loss: 0.2892 - val_accuracy: 0.8850 - val_loss: 0.2868
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 489s 865ms/step - accuracy: 0.9343 - loss: 0.1866 - val_accuracy: 0.8861 - val_loss: 0.2954


In [23]:
loss, acc = model.evaluate(X_test_pad, y_test)

print("Test Accuracy:", acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 34s 107ms/step - accuracy: 0.8861 - loss: 0.2897
Test Accuracy: 0.8860999941825867


In [24]:
def predict_sentiment(review):

    review = clean_text(review)

    seq = tokenizer.texts_to_sequences([review])

    padded = pad_sequences(seq, maxlen=max_len, padding='post')

    prediction = model.predict(padded)[0][0]

    print("\nReview:", review)
    print("Score:", prediction)

    if prediction >= 0.5:
        print("Sentiment: Positive 😊")
    else:
        print("Sentiment: Negative 😞")

In [25]:
predict_sentiment("This movie was absolutely amazing and I loved it")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 489ms/step

Review: this movie was absolutely amazing and i loved it
Score: 0.9817985
Sentiment: Positive 😊


In [26]:
predict_sentiment("This movie was not good and I didn't like it at all")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step

Review: this movie was not_good and i didn't_like it at all
Score: 0.28752047
Sentiment: Negative 😞


In [27]:
predict_sentiment("This movie is very entertaining! ")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step

Review: this movie is very entertaining  
Score: 0.94312286
Sentiment: Positive 😊


In [29]:
predict_sentiment("This movie look good only by seen its characters but the story is boring.overall a average movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step

Review: this movie look good only by seen its characters but the story is boring overall a average movie
Score: 0.10004951
Sentiment: Negative 😞


In [32]:
predict_sentiment("one of the good movie")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step

Review: one of the good movie
Score: 0.89911926
Sentiment: Positive 😊


In [27]:
predict_sentiment("This movie look good only by seen its characters but the story is boring.overall a avergae movie")